# 06 — Build the supplementary material (CPU, a few minutes)
Assembles `TMLR_v3/supplementary_v3.zip` from your project folder and verifies it before you upload it.

* **Registered analysis files are copied byte-for-byte**, so their hashes match `prereg.json`.
* **Data files are sanitised:** absolute Drive paths are replaced by `<DRIVE>/`.
* **Notebooks are included with outputs stripped**, because outputs can contain paths, account names and tokens.
* **The previous (pilot) supplementary archive is included unchanged** under `pilot/`.
* **Anonymity scan:** every text file is checked for the terms you list in `TMLR_v3/anonymity_terms.txt` (author
  names, institution; kept outside the archive) and for e-mail addresses. Any hit stops the build.
* **Clean-copy verification:** the finished zip is extracted into a fresh folder and `code/verify_v3.py` must pass
  there, including every number quoted in the paper.

**Before running:** upload to `MyDrive/TMLR_v3/` the previous supplementary zip
(`10271_Diversity_Is_Not_Enough__Supplementary_Material_.zip`), and put copies of notebooks 00–08 in
`MyDrive/TMLR_v3/notebooks/` (or leave them in `Colab Notebooks`).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys
PROJECT = "/content/drive/MyDrive/TMLR_v3"
CODE = os.path.join(PROJECT, "code")
assert os.path.exists(os.path.join(CODE, "config.py")), "run 00_setup_and_prereg.ipynb first"
if CODE not in sys.path:
    sys.path.insert(0, CODE)
import importlib, config as CFG
importlib.reload(CFG)
print("Drive:", CFG.DRIVE, "| project:", CFG.PROJECT)

In [ ]:
# %%writefile does not create folders, so make the ones the next cells write into
for d in ("code", "supplement_templates", "notebooks"):
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)
print("folders ready")

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/verify_v3.py
"""Verification gate for the multi-pool study (supplementary material, v3).

    python code/verify_v3.py --root .          # exit 0 = every check passed

Checks, in order. Each is reported PASS / FAIL; any FAIL exits non-zero.

  1. registration   the SHA-256 of every registered analysis file in code/ equals
                    the hash recorded in registration/prereg.json, so the code in
                    this archive is the code that was registered.
  2. records        every per-pool record carries the registration hash and the
                    manifest hash of the pool it was computed on.
  3. battery        recomputing estimands A-F from results/pools/*.json with the
                    registered code reproduces results/battery.json, every field to
                    within floating-point rounding (rtol 1e-9); differing fields are listed.
  4. tables         rebuilding the paper tables from those records reproduces
                    results/paper_tables.json (registered estimands and the
                    exploratory analyses), asserting every internal identity.
  5. claims         the numbers quoted in the paper equal the recomputed values at
                    the precision printed in the paper.

Nothing here needs a GPU, torch or the features: the per-pool records contain
every quantity the estimands are computed from. Regenerating the records from
public data is what notebooks/01-04 do.
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import json
import os
import sys

# Numbers as printed in the paper: (subset, path into battery.json, printed value, decimals)
CLAIMS = [
    ("all", ["A_matched_null_primary_only", "cluster_robust", "estimate_pp"], -0.34, 2),
    ("all", ["A_matched_null_primary_only", "cluster_robust", "ci_low"], -0.84, 2),
    ("all", ["A_matched_null_primary_only", "cluster_robust", "ci_high"], 0.16, 2),
    ("all", ["A_matched_null_primary_only", "cluster_robust", "G"], 18, 0),
    ("all", ["A_matched_null_all_pairs", "cluster_robust", "estimate_pp"], -0.10, 2),
    ("all", ["A_matched_null_all_pairs", "cluster_robust", "n"], 65, 0),
    ("all", ["C_equivalence", "margin_0.5", "equivalent"], True, None),
    ("all", ["C_equivalence", "margin_0.25", "equivalent"], False, None),
    ("image", ["C_equivalence", "margin_0.25", "equivalent"], True, None),
    ("pathology", ["C_equivalence", "margin_0.25", "equivalent"], True, None),
    ("text", ["A_matched_null_all_pairs", "cluster_robust", "estimate_pp"], -0.92, 2),
    ("all", ["D_selection_regret", "per_rule", "cka_diversity", "mean_regret_pp"], 1.76, 2),
    ("all", ["D_selection_regret", "per_rule", "competence", "mean_regret_pp"], 0.27, 2),
    ("all", ["D_selection_regret", "per_rule", "disagreement_with_base", "mean_regret_pp"], 1.92, 2),
    ("all", ["D_selection_regret", "per_rule", "oracle_headroom", "mean_regret_pp"], 0.28, 2),
    ("all", ["B_dose_response", "mde_80pct_power"], 3.51, 2),
    ("all", ["E_mechanism_on_real", "joint_model", "cka", "coef_pp_per_unit"], 1.94, 2),
]

RESULTS = []


def report(name, ok, detail=""):
    RESULTS.append((name, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  -- {detail}" if detail else ""), flush=True)


def sha256(path):
    with open(path, "rb") as fh:
        return hashlib.sha256(fh.read()).hexdigest()


def dig(d, path):
    for k in path:
        d = d[k]
    return d


def canon(obj):
    return json.dumps(obj, sort_keys=True, default=float)


RTOL, ATOL = 1e-9, 1e-12   # floating-point reproducibility across processes / BLAS builds


def diff(a, b, path="", out=None):
    """Every leaf where a and b differ beyond floating-point rounding.
    Returns a list of (path, saved, recomputed). Numbers compare with
    |a - b| <= ATOL + RTOL * |b|; NaN equals NaN; everything else exactly."""
    import math
    out = [] if out is None else out
    if isinstance(a, dict) and isinstance(b, dict):
        for k in sorted(set(a) | set(b)):
            if k not in a or k not in b:
                out.append((f"{path}/{k}", a.get(k, "<absent>"), b.get(k, "<absent>")))
            else:
                diff(a[k], b[k], f"{path}/{k}", out)
    elif isinstance(a, list) and isinstance(b, list):
        if len(a) != len(b):
            out.append((path, f"len {len(a)}", f"len {len(b)}"))
        else:
            for i, (x, y) in enumerate(zip(a, b)):
                diff(x, y, f"{path}[{i}]", out)
    elif isinstance(a, bool) or isinstance(b, bool) or not (
            isinstance(a, (int, float)) and isinstance(b, (int, float))):
        if a != b:
            out.append((path, a, b))
    else:
        fa, fb = float(a), float(b)
        if math.isnan(fa) and math.isnan(fb):
            return out
        if not (abs(fa - fb) <= ATOL + RTOL * abs(fb)):
            out.append((path, a, b))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default=".")
    ap.add_argument("--no-claims", action="store_true",
                    help="skip check 5 (for archives built from synthetic test data)")
    a = ap.parse_args()
    root = os.path.abspath(a.root)
    code = os.path.join(root, "code")
    res = os.path.join(root, "results")
    sys.path.insert(0, code)

    # 1. registration ------------------------------------------------------
    with open(os.path.join(root, "registration", "prereg.json")) as fh:
        reg = json.load(fh)
    for f, h in sorted(reg["file_sha256"].items()):
        p = os.path.join(code, f)
        ok = os.path.exists(p) and sha256(p) == h
        report(f"registration: {f}", ok, "" if ok else "hash differs from prereg.json")
    blob = json.dumps({k: reg[k] for k in ("params", "datasets", "encoders", "file_sha256")},
                      sort_keys=True).encode()
    report("registration: registration_hash", hashlib.sha256(blob).hexdigest() == reg["registration_hash"])

    # 2. records -------------------------------------------------------------
    pools = {}
    for f in sorted(glob.glob(os.path.join(res, "pools", "*.json"))):
        with open(f) as fh:
            r = json.load(fh)
        pools[r["pool"]] = r
    report("records: pool records present", len(pools) > 0, f"{len(pools)} pools")
    man_dir = os.path.join(root, "manifests")
    for name, r in pools.items():
        ok = r.get("prereg", {}).get("registration_hash") == reg["registration_hash"]
        mp = os.path.join(man_dir, f"{name}.json")
        if os.path.exists(mp):
            with open(mp) as fh:
                ok = ok and json.load(fh)["hash"] == r.get("manifest_hash")
        else:
            ok = False
        report(f"records: {name}", ok, "" if ok else "registration or manifest hash mismatch")

    # 3. battery -------------------------------------------------------------
    import analyse_all as A
    with open(os.path.join(res, "battery.json")) as fh:
        bat = json.load(fh)
    import config as CFG
    order = [ds for dom in ("pathology", "vision", "text") for ds in CFG.DOMAINS[dom][0]]
    records = [pools[p] for p in order if p in pools]     # the order notebook 04 used
    records += [pools[p] for p in pools if p not in order]
    again = A.battery(records, bat["prereg"])
    for sub in A.SUBSETS:
        x = json.loads(canon({k: v for k, v in bat.get(sub, {}).items() if k != "pools"}))
        y = json.loads(canon({k: v for k, v in again.get(sub, {}).items() if k != "pools"}))
        d = diff(x, y)
        exact = canon(x) == canon(y)
        # The random-intercept model on the PRIMARY rows is degenerate by design: one
        # observation per pool leaves no within-pool variance, so REML returns a standard
        # error that is numerically zero and whose last digits are optimiser noise. The paper
        # (Appendix B) states these rows are not estimable and does not report them. For them
        # we verify the documented degeneracy instead of comparing noise.
        DEGEN = "/A_matched_null_primary_only/mixed_effects/"
        degen = [t for t in d if t[0].startswith(DEGEN)]
        rest = [t for t in d if not t[0].startswith(DEGEN)]
        degen_ok, degen_note = True, ""
        if degen:
            import math
            def tiny(v):
                try:
                    v = float(v)
                    return math.isnan(v) or abs(v) < 1e-6
                except (TypeError, ValueError):
                    return False
            mx = x.get("A_matched_null_primary_only", {}).get("mixed_effects", {})
            my = y.get("A_matched_null_primary_only", {}).get("mixed_effects", {})
            cr = x.get("A_matched_null_primary_only", {}).get("cluster_robust", {})
            degen_ok = (tiny(mx.get("se")) and tiny(my.get("se"))
                        and abs(float(mx.get("estimate")) - float(my.get("estimate"))) < 1e-9
                        and abs(float(mx.get("estimate")) - float(cr.get("estimate_pp"))) < 1e-9)
            degen_note = (f"; primary-row random-intercept model degenerate as documented "
                          f"(SE saved {mx.get('se')!r}, recomputed {my.get('se')!r}; not reported, "
                          f"Appendix B)" if degen_ok else
                          "; primary-row random-intercept model is NOT degenerate as documented")
        ok = not rest and degen_ok
        if exact:
            detail = "bit-identical"
        elif ok:
            detail = f"identical to within rtol {RTOL:g}" + degen_note
        else:
            detail = ((f"{len(rest)} field(s) differ, e.g. " + "; ".join(
                f"{p}: saved {v1!r} vs recomputed {v2!r}" for p, v1, v2 in rest[:3]))
                      if rest else "") + degen_note.lstrip("; ")
        report(f"battery: subset '{sub}' recomputes", ok, detail)

    # 4. tables --------------------------------------------------------------
    import report as R
    pt = os.path.join(res, "paper_tables.json")
    if os.path.exists(pt):
        with open(pt) as fh:
            saved = json.load(fh)
        import tempfile, shutil
        tmp = tempfile.mkdtemp()
        shutil.copytree(os.path.join(res, "pools"), os.path.join(tmp, "pools"))
        shutil.copy(os.path.join(res, "battery.json"), tmp)
        if os.path.exists(os.path.join(res, "fusion_fit_log.csv")):
            shutil.copy(os.path.join(res, "fusion_fit_log.csv"), tmp)
        try:
            T, _ = R.build(tmp)          # asserts every recomputation equals the registered value
            report("tables: internal identities", True)
            d = diff(saved, json.loads(canon(T)))
            report("tables: paper_tables.json reproduces", not d,
                   "" if not d else f"{len(d)} field(s) differ, e.g. {d[:2]}")
        except AssertionError as e:
            report("tables: internal identities", False, str(e))
        shutil.rmtree(tmp, ignore_errors=True)
    else:
        report("tables: paper_tables.json present", False)

    # 5. claims --------------------------------------------------------------
    if not a.no_claims:
        for sub, path, printed, nd in CLAIMS:
            try:
                v = dig(bat[sub], path)
                ok = (v == printed) if nd is None else (round(float(v), nd) == printed)
            except Exception as e:
                v, ok = f"missing ({e})", False
            report(f"claim: {sub}/{'/'.join(path)} = {printed}", ok, "" if ok else f"recomputed {v}")

    n_fail = sum(1 for _, ok in RESULTS if not ok)
    print(f"\n{len(RESULTS) - n_fail}/{len(RESULTS)} checks passed")
    sys.exit(1 if n_fail else 0)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/build_supplement.py
"""Assemble the v3 supplementary archive from the project folder on Drive.

The registered analysis files are copied byte-for-byte from TMLR_v3/code, so
their SHA-256 hashes match registration/prereg.json by construction. Data files
are sanitised (absolute Drive paths replaced by placeholders), every text file is
scanned for identifying strings, every file is hashed into MANIFEST.json, and the
archive is verified in a clean temporary copy before it is reported as ready.
"""

from __future__ import annotations

import glob
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import config as CFG

TEXT_EXT = {".py", ".md", ".json", ".csv", ".txt", ".ipynb", ".tex", ".yml", ".yaml", ".cfg"}
PATH_RE = re.compile(r"/content/drive/My ?Drive/|/content/v3_local/|/content/")
RESULT_FILES = ["battery.json", "summary.md", "paper_tables.md", "paper_tables.json",
                "fusion_fit_log.csv", "v2_consistency.csv", "resnet50_provenance_mhist.csv"]


def _sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def _sanitise_text(s):
    return PATH_RE.sub(lambda m: "<DRIVE>/" if "drive" in m.group(0) else "<LOCAL>/", s)


def _copy_sanitised(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(src, encoding="utf-8", errors="strict") as fh:
        s = fh.read()
    with open(dst, "w", encoding="utf-8") as fh:
        fh.write(_sanitise_text(s))


def _strip_notebook(src, dst):
    """Copy a notebook with all outputs and execution counts removed (outputs can
    contain account names, paths and tokens)."""
    with open(src) as fh:
        nb = json.load(fh)
    for c in nb.get("cells", []):
        if c.get("cell_type") == "code":
            c["outputs"] = []
            c["execution_count"] = None
        c.get("metadata", {}).pop("colab", None)
        c.get("metadata", {}).pop("outputId", None)
        c.get("metadata", {}).pop("id", None)
    nb["metadata"].pop("colab", None)
    nb["metadata"].pop("widgets", None)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, "w") as fh:
        json.dump(nb, fh, indent=1)


def stage(stage_dir, notebook_dir=None, pilot_zip=None, readme=None, requirements=None):
    if os.path.exists(stage_dir):
        shutil.rmtree(stage_dir)
    os.makedirs(stage_dir)
    log = {"copied": [], "excluded": [], "notes": []}

    # code: byte-for-byte (the registered files must keep their hashes)
    os.makedirs(os.path.join(stage_dir, "code"))
    for f in sorted(glob.glob(os.path.join(CFG.CODE_DIR, "*.py"))):
        shutil.copy2(f, os.path.join(stage_dir, "code", os.path.basename(f)))
    tests = os.path.join(CFG.PROJECT, "tests")
    if os.path.isdir(tests):
        shutil.copytree(tests, os.path.join(stage_dir, "tests"),
                        ignore=shutil.ignore_patterns("__pycache__"))

    # registration (+ Google Drive's own record of the file, if notebook 06 fetched it)
    os.makedirs(os.path.join(stage_dir, "registration"))
    shutil.copy2(CFG.PREREG_PATH, os.path.join(stage_dir, "registration", "prereg.json"))
    dm = os.path.join(CFG.PROJECT, "registration_drive_metadata.json")
    if os.path.exists(dm):
        shutil.copy2(dm, os.path.join(stage_dir, "registration", "drive_metadata.json"))
    else:
        log["notes"].append("no Drive metadata for prereg.json (optional cell in notebook 06 not run)")

    # results
    rd = CFG.RESULTS_DIR
    for f in RESULT_FILES + [os.path.basename(p) for p in glob.glob(os.path.join(rd, "feature_sanity_*.csv"))]:
        p = os.path.join(rd, f)
        if os.path.exists(p):
            _copy_sanitised(p, os.path.join(stage_dir, "results", f))
        else:
            log["notes"].append(f"results/{f} not found (skipped)")
    for p in sorted(glob.glob(os.path.join(rd, "pools", "*.json"))):
        _copy_sanitised(p, os.path.join(stage_dir, "results", "pools", os.path.basename(p)))
    for p in sorted(glob.glob(os.path.join(rd, "pools", "*_logits.npz"))):
        log["excluded"].append({"file": f"results/pools/{os.path.basename(p)}",
                                "bytes": os.path.getsize(p),
                                "reason": "held-out logits; regenerated by notebook 04"})
    rv = os.path.join(rd, "reviewer")
    if os.path.isdir(rv):                      # exploratory reviewer analyses (notebook 07)
        for p in sorted(glob.glob(os.path.join(rv, "*"))):
            if p.endswith((".json", ".md", ".csv")):
                _copy_sanitised(p, os.path.join(stage_dir, "results", "reviewer", os.path.basename(p)))
            elif os.path.isfile(p):
                os.makedirs(os.path.join(stage_dir, "results", "reviewer"), exist_ok=True)
                shutil.copy2(p, os.path.join(stage_dir, "results", "reviewer", os.path.basename(p)))
    figs = os.path.join(rd, "figures")
    if os.path.isdir(figs):
        shutil.copytree(figs, os.path.join(stage_dir, "results", "figures"))

    # manifests (row order, labels, groups, hashes) and feature provenance
    n_texts = 0
    for p in sorted(glob.glob(os.path.join(CFG.MANIFEST_DIR, "*"))):
        dst = os.path.join(stage_dir, "manifests", os.path.basename(p))
        if p.endswith(".json"):
            with open(p) as fh:
                m = json.load(fh)
            if "texts" in m:
                # Dataset text is third-party content: not redistributed. The manifest hash
                # covers (uid, label) only, and uids index the public split, so rows are
                # still reproducible exactly from the named source.
                n_texts += len(m.pop("texts"))
                m["texts_omitted"] = "dataset text not redistributed; regenerate from 'source'"
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with open(dst, "w") as fh:
                fh.write(_sanitise_text(json.dumps(m)))
        else:
            _copy_sanitised(p, dst)
    if n_texts:
        log["excluded"].append({"file": "manifests/<text dataset>.json: field 'texts'", "count": n_texts,
                                "reason": "third-party dataset text not redistributed; "
                                          "regenerated by notebook 03 from the public source"})
    for p in sorted(glob.glob(os.path.join(CFG.FEAT_DIR, "*", "*.json"))):
        ds = os.path.basename(os.path.dirname(p))
        _copy_sanitised(p, os.path.join(stage_dir, "features_provenance", ds, os.path.basename(p)))
    n_feat = len(glob.glob(os.path.join(CFG.FEAT_DIR, "*", "*.npy")))
    log["excluded"].append({"file": "features/<dataset>/<encoder>.npy", "count": n_feat,
                            "reason": "frozen features (~10 GB); regenerated by notebooks 01-03"})

    # extraction run logs (status of every dataset x encoder, timestamped by file name)
    for p in sorted(glob.glob(os.path.join(CFG.LOG_DIR, "*.json"))):
        _copy_sanitised(p, os.path.join(stage_dir, "logs", os.path.basename(p)))

    # notebooks, outputs stripped
    dirs = [d for d in ([notebook_dir] if isinstance(notebook_dir, str) else list(notebook_dir or []))
            + [os.path.join(CFG.PROJECT, "notebooks"), os.path.join(CFG.DRIVE, "Colab Notebooks"), CFG.PROJECT]
            if d and os.path.isdir(d)]
    found = {}
    for d in dirs:                                   # first folder that has a given notebook wins
        for p in sorted(glob.glob(os.path.join(d, "0[0-8]_*.ipynb"))):
            found.setdefault(os.path.basename(p)[:2], p)
    for k, p in sorted(found.items()):
        _strip_notebook(p, os.path.join(stage_dir, "notebooks", os.path.basename(p)))
    log["notebooks_missing"] = [f"{i:02d}" for i in range(9) if f"{i:02d}" not in found]
    if log["notebooks_missing"]:
        log["notes"].append("notebooks missing: " + ", ".join(log["notebooks_missing"]))

    # the pilot's archive, unchanged
    if pilot_zip and os.path.exists(pilot_zip):
        with zipfile.ZipFile(pilot_zip) as z:
            names = [m for m in z.namelist() if not (m.startswith("__MACOSX") or
                     m.endswith(".DS_Store") or m.endswith("/"))]
            tops = {m.split("/")[0] for m in names}
            strip = len(tops) == 1 and all("/" in m for m in names)   # single wrapping folder
            for m in names:
                rel = m.split("/", 1)[1] if strip else m
                out = os.path.join(stage_dir, "pilot", rel)
                os.makedirs(os.path.dirname(out), exist_ok=True)
                with z.open(m) as src, open(out, "wb") as dst:
                    shutil.copyfileobj(src, dst)
    else:
        log["notes"].append("pilot archive not given: pilot/ not included")

    if readme:
        with open(os.path.join(stage_dir, "README.md"), "w") as fh:
            fh.write(readme)
    if requirements:
        with open(os.path.join(stage_dir, "requirements.txt"), "w") as fh:
            fh.write(requirements)
    return log


DATASET_ID_FIELDS = ("uids", "reader", "texts", "classes", "groups", "extra")


def _scannable(path, rel):
    """Text of a staged file as it should be scanned. Row identifiers, file names
    and class names copied from public datasets are third-party content (e.g. NCT
    image names, newsgroup host names), so they are excluded; everything we wrote
    is scanned in full."""
    if rel.startswith("manifests" + os.sep):
        if path.endswith(".csv"):
            return ""                      # row, uid, label, group: all dataset-derived
        try:
            m = json.load(open(path))
            return json.dumps({k: v for k, v in m.items() if k not in DATASET_ID_FIELDS})
        except Exception:
            pass
    return open(path, encoding="utf-8", errors="ignore").read()


def _parse_time(t):
    """Timestamps in provenance records are written by time.strftime on Colab,
    whose clock is UTC by default; the registration time is explicit UTC."""
    import datetime as dt
    if not t or not isinstance(t, str):
        return None
    t = t.strip().replace("Z", "+00:00")
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%dT%H:%M:%S"):
        try:
            d = dt.datetime.strptime(t, fmt)
            return d if d.tzinfo else d.replace(tzinfo=dt.timezone.utc)
        except ValueError:
            pass
    try:
        d = dt.datetime.fromisoformat(t)
        return d if d.tzinfo else d.replace(tzinfo=dt.timezone.utc)
    except ValueError:
        return None


def timeline(stage_dir):
    """Registration -> extraction -> analysis, from what is in the archive and on Drive.
    Writes registration/timeline.json and returns it. Any feature file saved before the
    registration time is listed under 'extracted_before_registration' (must be disclosed)."""
    import datetime as dt
    reg = json.load(open(os.path.join(stage_dir, "registration", "prereg.json")))
    t_reg = _parse_time(reg["registered_utc"])
    feats = []
    for p in sorted(glob.glob(os.path.join(stage_dir, "features_provenance", "*", "*.json"))):
        m = json.load(open(p))
        feats.append({"dataset": os.path.basename(os.path.dirname(p)),
                      "encoder": os.path.splitext(os.path.basename(p))[0],
                      "saved_utc": m.get("saved"), "_t": _parse_time(m.get("saved"))})
    # Features without their own timestamp (the text arm wrote none) are bounded from below by the build
    # time of the dataset manifest they were extracted from: extraction cannot precede the manifest.
    built = {}
    for p in glob.glob(os.path.join(stage_dir, "manifests", "*.json")):
        try:
            built[os.path.splitext(os.path.basename(p))[0]] = _parse_time(json.load(open(p)).get("built"))
        except Exception:
            pass
    for f in feats:
        f["manifest_built_utc"] = built.get(f["dataset"]).isoformat(timespec="seconds") if built.get(f["dataset"]) else None
        f["_lb"] = f["_t"] or built.get(f["dataset"])
    parsed = [f for f in feats if f["_t"] is not None]
    bounded = [f for f in feats if f["_t"] is None and f["_lb"] is not None]
    unknown = [f for f in feats if f["_lb"] is None]
    before = [f for f in feats if t_reg and f["_lb"] is not None and f["_lb"] < t_reg]
    logs = []
    for p in sorted(glob.glob(os.path.join(CFG.LOG_DIR, "extract_*.json"))):
        stamp = os.path.basename(p)[len("extract_"):-len(".json")]
        try:
            logs.append(dt.datetime.strptime(stamp, "%Y%m%d_%H%M%S").replace(tzinfo=dt.timezone.utc))
        except ValueError:
            pass
    bat = os.path.join(stage_dir, "results", "battery.json")
    t_bat = _parse_time(json.load(open(bat)).get("generated")) if os.path.exists(bat) else None
    iso = lambda d: d.isoformat(timespec="seconds") if d else None
    out = {
        "registration_utc": iso(t_reg),
        "registration_file_mtime_utc": iso(dt.datetime.fromtimestamp(os.path.getmtime(CFG.PREREG_PATH),
                                                                       tz=dt.timezone.utc)),
        "feature_files": len(feats), "feature_files_with_timestamp": len(parsed),
        "first_feature_saved_utc": iso(min(f["_t"] for f in parsed)) if parsed else None,
        "last_feature_saved_utc": iso(max(f["_t"] for f in parsed)) if parsed else None,
        "feature_files_bounded_by_manifest_build_time": len(bounded),
        "feature_files_without_any_time": len(unknown),
        "earliest_manifest_built_utc": (min(d for d in built.values() if d).isoformat(timespec="seconds")
                                        if any(built.values()) else None),
        "extracted_before_registration": [{k: v for k, v in f.items() if k not in ("_t", "_lb")} for f in before],
        "extraction_runs_utc": [iso(d) for d in logs],
        "extraction_runs_before_registration": [iso(d) for d in logs if t_reg and d < t_reg],
        "analysis_battery_generated_utc": iso(t_bat),
        "clock_note": "provenance and battery timestamps were written by time.strftime on Colab, whose clock is UTC "
                      "by default; the registration time is recorded in UTC",
    }
    dmp = os.path.join(stage_dir, "registration", "drive_metadata.json")
    if os.path.exists(dmp):
        dm = json.load(open(dmp))
        out["registration_drive_created_utc"] = dm.get("createdTime")
        out["registration_drive_md5_matches_shipped_file"] = dm.get("md5_matches_local")
    out["all_features_after_registration"] = bool(feats) and not before and not unknown
    with open(os.path.join(stage_dir, "registration", "timeline.json"), "w") as fh:
        json.dump(out, fh, indent=1)
    return out


def anonymity_scan(stage_dir, forbidden):
    """Whole-word, case-insensitive search for identifying terms and e-mail
    addresses in every text file of the archive."""
    hits = []
    pats = [(w, re.compile(r"(?<![A-Za-z0-9])" + re.escape(w.strip()) + r"(?![A-Za-z0-9])", re.I))
            for w in forbidden if w.strip()]
    emails = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
    for root, _, files in os.walk(stage_dir):
        for f in files:
            p = os.path.join(root, f)
            rel = os.path.relpath(p, stage_dir)
            if os.path.splitext(f)[1].lower() not in TEXT_EXT:
                continue
            s = _scannable(p, rel)
            for w, pat in pats:
                for m in pat.finditer(s):
                    ctx = s[max(0, m.start() - 40): m.end() + 40].replace("\n", " ")
                    hits.append((rel, w, ctx))
            if not rel.startswith("pilot" + os.sep):
                for m in emails.finditer(s):
                    if not m.group(0).endswith(("example.com", "anon.org")):
                        hits.append((rel, "email", m.group(0)))
    return hits


def write_manifest(stage_dir, log):
    files = []
    for root, _, fs in os.walk(stage_dir):
        for f in sorted(fs):
            p = os.path.join(root, f)
            rel = os.path.relpath(p, stage_dir)
            if rel == "MANIFEST.json":
                continue
            files.append({"file": rel, "bytes": os.path.getsize(p), "sha256": _sha(p)})
    man = {"built_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "n_files": len(files), "total_bytes": sum(f["bytes"] for f in files),
           "files": sorted(files, key=lambda f: f["file"]),
           "excluded": log["excluded"], "notes": log["notes"]}
    with open(os.path.join(stage_dir, "MANIFEST.json"), "w") as fh:
        json.dump(man, fh, indent=1)
    return man


def zip_dir(stage_dir, out_zip, top="supplementary"):
    if os.path.exists(out_zip):
        os.remove(out_zip)
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        for root, _, fs in os.walk(stage_dir):
            for f in sorted(fs):
                p = os.path.join(root, f)
                z.write(p, os.path.join(top, os.path.relpath(p, stage_dir)))
    return os.path.getsize(out_zip)


def verify_archive(out_zip, extra_args=()):
    tmp = tempfile.mkdtemp()
    with zipfile.ZipFile(out_zip) as z:
        z.extractall(tmp)
    root = os.path.join(tmp, "supplementary")
    env = dict(os.environ)
    env.pop("V3_DRIVE", None)
    r = subprocess.run([sys.executable, os.path.join(root, "code", "verify_v3.py"), "--root", root,
                        *extra_args], capture_output=True, text=True, env=env, cwd=root)
    shutil.rmtree(tmp, ignore_errors=True)
    return r.returncode, r.stdout, r.stderr

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/report.py
"""Paper tables from the SAVED results of notebook 04. Read-only: it never
re-fits a probe or a fusion and never changes battery.json.

Two kinds of output, kept strictly apart:

  REGISTERED   numbers read straight from battery.json (estimands A-F). The only
               change is presentation: equivalence is reported as the TIGHTEST
               margin cleared (summary.md printed max() of the cleared margins,
               i.e. the weakest claim -- a labelling bug in a registered file,
               corrected here at the reporting stage).
  EXPLORATORY  analyses that were not pre-registered (paired regret test,
               leave-one-pool-out). Every such table is labelled EXPLORATORY.

Every recomputation that should equal a registered number is asserted to.
"""

from __future__ import annotations

import csv
import glob
import json
import os

import numpy as np
from scipy import stats

import config as CFG
from inference import cluster_robust_ols

SUBSET_ORDER = ["all", "pathology", "vision", "text", "image"]
RULES = ["cka_diversity", "disagreement_with_base", "competence", "oracle_headroom"]
REVERSE = {"cka_diversity": False, "disagreement_with_base": True,
           "competence": True, "oracle_headroom": True}
KEY = {"cka_diversity": "cka", "disagreement_with_base": "disagreement",
       "competence": "indiv_acc", "oracle_headroom": "headroom_contrib"}


def load(results_dir=None):
    rd = results_dir or CFG.RESULTS_DIR
    with open(os.path.join(rd, "battery.json")) as fh:
        bat = json.load(fh)
    pools = {}
    for f in sorted(glob.glob(os.path.join(rd, "pools", "*.json"))):
        with open(f) as fh:
            r = json.load(fh)
        pools[r["pool"]] = r
    return bat, pools


def _subset_pools(bat, name):
    return bat[name].get("pools", [])


# ---------------------------------------------------------------- tables
def pools_table(pools):
    rows = []
    for name, r in pools.items():
        prim = next((p for p in r["matched_pairs"] if p.get("primary")), None)
        rows.append({
            "pool": name, "domain": r["domain"], "n": r["n_inputs"], "encoders": r["n_members"],
            "grouped": r["grouped_folds"], "anchor": r["anchor"], "base_member": r["base"][1],
            "matched_pairs": len(r["matched_pairs"]),
            "primary_pair": f"{prim['high']} vs {prim['low']}" if prim else "",
            "primary_cka_gap": round(prim["cka_gap"], 3) if prim else None,
            "primary_comp_gap_pp": round(prim["comp_gap"], 2) if prim else None,
            "primary_D_pp": round(prim["diff"], 2) if prim else None,
        })
    return rows


def estimand_A(bat):
    rows = []
    for s in SUBSET_ORDER:
        b = bat.get(s, {})
        for key, label in (("A_matched_null_primary_only", "primary (1 pair/pool)"),
                           ("A_matched_null_all_pairs", "all matched pairs")):
            a = b.get(key) or {}
            cr = a.get("cluster_robust")
            if not cr:
                rows.append({"subset": s, "analysis": label,
                             "note": a.get("not_estimable", "n/a")})
                continue
            me = a.get("mixed_effects", {})
            rows.append({"subset": s, "analysis": label, "D_pp": cr["estimate_pp"],
                         "ci_low": cr["ci_low"], "ci_high": cr["ci_high"], "p": cr["p"],
                         "pairs": cr["n"], "G": cr["G"], "reference": cr["reference"],
                         "mixed_effects_D": me.get("estimate"), "mixed_effects_p": me.get("p")})
    return rows


def estimand_C(bat):
    """TOST at the registered margins; tightest margin cleared (the correct label)."""
    rows = []
    for s in SUBSET_ORDER:
        c = bat.get(s, {}).get("C_equivalence") or {}
        if "not_estimable" in c or not c:
            rows.append({"subset": s, "note": c.get("not_estimable", "n/a")})
            continue
        margins = sorted((float(k.split("_")[1]) for k in c if k.startswith("margin_")), reverse=True)
        cleared = [m for m in margins if c[f"margin_{m}"]["equivalent"]]
        row = {"subset": s, "G": c["G"],
               "estimate_pp": c[f"margin_{margins[0]}"]["estimate"],
               "ci90_low": c[f"margin_{margins[0]}"]["ci90_low"],
               "ci90_high": c[f"margin_{margins[0]}"]["ci90_high"],
               "tightest_margin_cleared": min(cleared) if cleared else None,
               "summary_md_printed": c.get("largest_margin_cleared")}
        for m in margins:
            row[f"p_tost_{m}"] = c[f"margin_{m}"]["p_tost"]
        rows.append(row)
    return rows


def estimand_B_E(bat):
    rows = []
    for s in SUBSET_ORDER:
        b = bat.get(s, {})
        B = b.get("B_dose_response") or {}
        E = (b.get("E_mechanism_on_real") or {}).get("joint_model") or {}
        rows.append({
            "subset": s,
            "B_slope_pp_per_cka": B.get("slope_pp_per_unit_cka"), "B_p": B.get("p"),
            "B_mde80": B.get("mde_80pct_power"),
            "B_interpretable(MDE<=0.5)": (B.get("mde_80pct_power", 99) <= 0.5) if B else None,
            "E_cka_coef": (E.get("cka") or {}).get("coef_pp_per_unit"),
            "E_cka_p": (E.get("cka") or {}).get("p"),
            "E_disagreement_coef": (E.get("disagreement") or {}).get("coef_pp_per_unit"),
            "E_disagreement_p": (E.get("disagreement") or {}).get("p"),
            "E_acc_coef": (E.get("indiv_acc") or {}).get("coef_pp_per_unit"),
            "E_acc_p": (E.get("indiv_acc") or {}).get("p"),
            "E_G": (b.get("E_mechanism_on_real") or {}).get("G"),
        })
    return rows


def _regrets(cands):
    best = max(c["utility"] for c in cands)
    out = {}
    for r in RULES:
        fn = max if REVERSE[r] else min
        pick = fn(cands, key=lambda c: float(c[KEY[r]]))
        out[r] = best - pick["utility"]
    return out


def estimand_D(bat, pools):
    """Registered regret table + EXPLORATORY paired test CKA vs competence."""
    reg_rows, paired_rows = [], []
    for s in SUBSET_ORDER:
        D = (bat.get(s, {}).get("D_selection_regret") or {})
        pr = D.get("per_rule", {})
        for r in RULES:
            if r in pr:
                reg_rows.append({"subset": s, "criterion": r, **{k: pr[r][k] for k in
                                 ("mean_regret_pp", "median_regret_pp", "p90_regret_pp",
                                  "top1_agreement_with_oracle", "n_pools")}})
        names = [p for p in _subset_pools(bat, s) if p in pools]
        per = {p: _regrets([c for c in pools[p]["candidates"] if np.isfinite(c.get("utility", np.nan))])
               for p in names}
        if not per:
            continue
        for r in RULES:          # recomputation must equal the registered mean
            mine = float(np.mean([per[p][r] for p in per]))
            assert abs(mine - pr[r]["mean_regret_pp"]) < 1e-9, (s, r, mine, pr[r]["mean_regret_pp"])
        d = np.array([per[p]["cka_diversity"] - per[p]["competence"] for p in per])
        t = stats.ttest_1samp(d, 0.0) if len(d) > 1 else None
        try:
            w = stats.wilcoxon(d) if np.any(d != 0) and len(d) > 1 else None
        except ValueError:
            w = None
        paired_rows.append({
            "subset": s, "pools": len(d),
            "mean_extra_regret_cka_vs_competence_pp": float(d.mean()),
            "ci_low": float(d.mean() - stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else None,
            "ci_high": float(d.mean() + stats.t.ppf(0.975, len(d) - 1) * d.std(ddof=1) / np.sqrt(len(d))) if len(d) > 1 else None,
            "t_p": float(t.pvalue) if t is not None else None,
            "wilcoxon_p": float(w.pvalue) if w is not None else None,
            "cka_worse": int((d > 1e-9).sum()), "tie": int((np.abs(d) <= 1e-9).sum()),
            "cka_better": int((d < -1e-9).sum()),
        })
    return reg_rows, paired_rows


def leave_one_pool_out(bat, pools, subset):
    """EXPLORATORY: estimand A recomputed with each pool dropped in turn.
    The full-sample recomputation is asserted equal to the registered value."""
    names = [p for p in _subset_pools(bat, subset) if p in pools]
    out = []
    for key, prim in (("A_matched_null_all_pairs", False), ("A_matched_null_primary_only", True)):
        def rows(excl=None):
            rr = [m for p in names if p != excl for m in pools[p]["matched_pairs"]
                  if np.isfinite(m["diff"]) and (m.get("primary") or not prim)]
            return np.array([m["diff"] for m in rr]), np.array([m["pool"] for m in rr])
        y, cl = rows()
        if len(np.unique(cl)) < 2:
            continue
        full = cluster_robust_ols(y, cl)
        reg = bat[subset][key]["cluster_robust"]
        assert abs(float(full["estimate"][0]) - reg["estimate_pp"]) < 1e-9
        assert abs(float(full["ci_low"][0]) - reg["ci_low"]) < 1e-9
        for p in [None] + names:
            y, cl = rows(p)
            if len(np.unique(cl)) < 2:
                continue
            r = cluster_robust_ols(y, cl)
            out.append({"subset": subset, "analysis": "primary" if prim else "all pairs",
                        "dropped": p or "(none)", "D_pp": float(r["estimate"][0]),
                        "ci_low": float(r["ci_low"][0]), "ci_high": float(r["ci_high"][0]),
                        "p": float(r["p"][0]), "G": int(r["G"]), "pairs": int(r["n"])})
    return out


def fit_caps(results_dir=None):
    p = os.path.join(results_dir or CFG.RESULTS_DIR, "fusion_fit_log.csv")
    if not os.path.exists(p):
        return []
    agg = {}
    with open(p) as fh:
        for r in csv.DictReader(fh):
            a = agg.setdefault(r["pool"], {"pool": r["pool"], "classes": int(r["C"]),
                                           "fits": 0, "capped": 0})
            a["fits"] += 1
            a["capped"] += r["converged"] != "True"
    for a in agg.values():
        a["capped_pct"] = round(100 * a["capped"] / a["fits"], 1)
    return list(agg.values())


# ---------------------------------------------------------------- writing
def _fmt(v, nd=2):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        return f"{v:.3g}" if abs(v) < 0.001 and v != 0 else f"{v:.{nd}f}"
    return str(v)


def md_table(rows, cols=None):
    if not rows:
        return "_(none)_\n"
    cols = cols or list(dict.fromkeys(k for r in rows for k in r))
    L = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    L += ["| " + " | ".join(_fmt(r.get(c)) for c in cols) + " |" for r in rows]
    return "\n".join(L) + "\n"


def build(results_dir=None):
    rd = results_dir or CFG.RESULTS_DIR
    bat, pools = load(rd)
    T = {"pools": pools_table(pools), "A": estimand_A(bat), "C": estimand_C(bat),
         "BE": estimand_B_E(bat)}
    T["D_registered"], T["D_paired_exploratory"] = estimand_D(bat, pools)
    T["LOO_text_exploratory"] = leave_one_pool_out(bat, pools, "text") if "text" in bat else []
    T["LOO_all_exploratory"] = leave_one_pool_out(bat, pools, "all")
    T["fit_caps"] = fit_caps(rd)
    md = [f"# v3 paper tables\n\nRegistered {bat['prereg']['registered_utc']} "
          f"(hash {bat['prereg']['registration_hash'][:12]}); deviations: "
          f"{bat['prereg']['deviations'] or 'none'}.\n",
          "## Pools\n", md_table(T["pools"]),
          "## A. Matched-competence null (REGISTERED; D = utility(higher-CKA) − utility(lower-CKA), pp)\n",
          md_table(T["A"]),
          "## C. Equivalence, TOST (REGISTERED; all matched pairs, cluster-robust)\n",
          "`tightest_margin_cleared` is the claim to make. `summary_md_printed` shows what summary.md "
          "reported (the largest margin cleared, i.e. the weakest claim).\n\n", md_table(T["C"]),
          "## B and E. Dose-response and mechanism (REGISTERED)\n", md_table(T["BE"]),
          "## D. Selection regret (REGISTERED)\n", md_table(T["D_registered"]),
          "## D′. Paired regret, CKA diversity minus competence (EXPLORATORY — not a registered test)\n",
          md_table(T["D_paired_exploratory"]),
          "## Leave-one-pool-out, text subset (EXPLORATORY)\n", md_table(T["LOO_text_exploratory"]),
          "## Leave-one-pool-out, all pools (EXPLORATORY)\n", md_table(T["LOO_all_exploratory"]),
          "## Secondary fusion rule: fits reaching the registered 500-iteration cap\n",
          "Pools analysed before the fit log existed (the pathology pools and DTD, Aircraft, EuroSAT, Pets) "
          "are not listed.\n\n", md_table(T["fit_caps"])]
    text = "\n".join(md)
    with open(os.path.join(rd, "paper_tables.md"), "w") as fh:
        fh.write(text)
    with open(os.path.join(rd, "paper_tables.json"), "w") as fh:
        json.dump(T, fh, indent=1, default=float)
    return T, text

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/supplement_templates/README.md
# Supplementary Material

*Diversity Is Not Enough: Representational Divergence Is a Poor Guide to Fusion Utility*

This archive contains the code, registration, per-pool records and verification
gate for the pre-registered twenty-pool study (main text Sections 3–4,
Appendices B–F), the exploratory analyses (Section 4.6, Appendix G,
`notebooks/07_reviewer_analyses.ipynb`, `results/reviewer/`), and, unchanged in
`pilot/`, the archive of the four-encoder pilot and the controlled study
(Sections 5–6, Appendices H–I and the remaining appendices).

---

## Quick start

```bash
pip install -r requirements.txt
python code/verify_v3.py --root .     # exit 0 = every check passed
```

This runs on CPU in about a minute and needs neither a GPU nor the features. The exploratory
notebooks (07, 08) cannot run from this archive alone: they need the held-out logits and
features, which notebooks 01-04 regenerate from public data. It
checks, and fails loudly on any mismatch, that:

1. **registration**: every registered analysis file in `code/` has the SHA-256
   recorded in `registration/prereg.json`, so this is the code that was registered;
2. **records**: every per-pool record carries the registration hash and the hash of
   the dataset manifest it was computed on;
3. **battery**: recomputing estimands A–F from `results/pools/*.json` reproduces
   `results/battery.json` exactly;
4. **tables**: rebuilding the paper tables reproduces `results/paper_tables.json`,
   including the exploratory paired-regret and leave-one-pool-out analyses;
5. **claims**: every registered number quoted in the paper's abstract and Section 4
   equals the recomputed value at the printed precision. (The exploratory numbers are
   produced by `notebooks/07_reviewer_analyses.ipynb`, which also asserts that it
   reproduces the registered utilities and regrets before running.)

To verify the pilot, see `pilot/README.md` (`python pilot/code/verify_claims.py --root pilot`).

---

## Timeline

`registration/timeline.json` puts the evidence in one place:
- the registration time and the registration file's own timestamps;
- every feature file's extraction time, from the per-encoder records in `features_provenance/`;
- the extraction runs in `logs/`;
- the time the analysis was generated.

No multi-pool feature, logit, accuracy or utility existed when the analysis was registered. The pilot's results
did exist, shaped the design, and are disclosed in paper Appendix B; excluding the six datasets the pilot shares
with this study leaves every conclusion unchanged (Section 4.6).

## The registration

`registration/prereg.json` was written before the twenty-pool analysis was run.
It fixes:

- the probe and the fold structure, including grouping by slide or patient;
- the anchor rule, base rule and matching thresholds;
- the fusion rules, the five estimands, the equivalence margins and the
  interpretability criterion for the dose–response estimand;
- the subsets, and the dataset and encoder lists.

It also records the hash of every analysis file. The analysis
(`notebooks/04_analysis.ipynb`) refuses to run if any of them differs. It ran
with no declared deviation, as every record in `results/pools/` states.

Two points concern presentation only (paper Appendix B):

- `results/summary.md`, generated automatically, labels the equivalence estimand
  with the *largest* margin cleared. The paper reports the *tightest*, read from
  the same registered tests in `battery.json`.
- The random-intercept robustness model is degenerate for the primary analysis,
  which has one observation per pool, so its p-values for those rows are not
  meaningful and are not reported.

---

## Contents

```
code/                    analysis, extraction and verification code
                         (registered files are listed in registration/prereg.json)
tests/                   unit and end-to-end tests (python tests/test_core.py, ...)
notebooks/               00 setup and registration, 01-03 feature extraction,
                         04 analysis, 05 paper tables, 06 this archive (outputs stripped)
registration/prereg.json the registration
registration/timeline.json  registration -> extraction -> analysis: every feature file
                         postdates the registration (by its own timestamp, or for the
                         text arm by its manifest's build time)
registration/drive_metadata.json  (if present) Google Drive's record of prereg.json:
                         creation time and an MD5 checksum matching the shipped file
logs/                    extraction run logs (status of every dataset x encoder)
manifests/<dataset>.json row order, labels, slide/patient groups and a hash of
                         (uid, label) for every dataset; absolute paths are
                         replaced by <DRIVE>/ placeholders. Text manifests omit
                         the dataset text itself (third-party content); uids index
                         the named public split, so rows are reproducible exactly
features_provenance/     for every (dataset, encoder): encoder source, weights
                         checksum, normalisation, pooling, load report and
                         feature statistics
results/pools/*.json     the complete per-pool record: per-fold accuracies, CKA,
                         anchor, base, every candidate's per-fold utility under
                         all three fusion rules, every pair and its category
results/battery.json     estimands A-F for every registered subset
results/paper_tables.*   the tables of the paper, built from the records
results/fusion_fit_log.csv  every secondary-rule fit (iterations, convergence)
results/v2_consistency.csv, resnet50_provenance_mhist.csv   Appendix D
pilot/                   the pilot's archive, unchanged
MANIFEST.json            every file with its SHA-256, and every exclusion
```

## What is not included, and how to regenerate it

The frozen features (about 10 GB) and the per-pool held-out logits are not
included because of their size. Both are regenerated from public data by the
notebooks:

- `notebooks/01_pathology_features.ipynb`, `02_vision_features.ipynb` and
  `03_text_features.ipynb` extract the features. All datasets and encoders are
  public; six pathology encoders require accepting a licence on Hugging Face.
- `notebooks/04_analysis.ipynb` regenerates the per-pool records and the logits.

Every extracted feature file is checked against the shipped manifest hash, so
regenerated features are aligned to the same rows.

Extraction details that matter for reproduction (paper Appendix C):

- **One decoded copy per dataset.** Every encoder in a pool reads the same copy,
  decoded at 224 px with Resize and CenterCrop.
- **CTransPath** runs in a separate process under timm 0.5.4. It asserts a complete
  load of the checkpoint, including the 16 convolutional-stem tensors whose silent
  omission is documented in `pilot/README.md`.
- **Hibou-B** runs in a separate process under transformers 4.46.3, because its
  custom code needs it.
- **The probe** is solved to its optimum by full-batch L-BFGS.
  `tests/test_core.py` checks it against a tightly converged scikit-learn reference.

## Reproducing the analysis from features

With the features in place (folder layout in `notebooks/00_setup_and_prereg.ipynb`), run
`notebooks/04_analysis.ipynb`. It checks the registration first, reuses nothing that
was computed on different features, and writes `results/`; `code/verify_v3.py`
then verifies the new results exactly as above.

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/supplement_templates/requirements.txt
# Verification gate (code/verify_v3.py) and the analysis: CPU only.
numpy>=1.26
scipy>=1.11
scikit-learn>=1.4
statsmodels>=0.14
pandas>=2.0
matplotlib>=3.8

# Only for regenerating per-pool records (probes run on GPU if available)
torch>=2.3

# Only for regenerating features (notebooks 01-03)
# timm>=1.0, transformers, sentence-transformers, datasets, h5py, openpyxl
# CTransPath: timm==0.5.4 (installed in an isolated folder automatically)
# Hibou-B: transformers==4.46.3 (installed in an isolated folder automatically)

# The pilot archive (pilot/) has its own pinned requirements: pilot/requirements.txt

## Settings

In [ ]:
import glob
# the previous (pilot) supplementary archive: any zip with "supplementary" in its name, in TMLR_v3/
# or in MyDrive/, other than the one this notebook writes
_cands = [p for d in (CFG.PROJECT, CFG.DRIVE) for p in glob.glob(os.path.join(d, "*.zip"))
          if "supplementary" in os.path.basename(p).lower()
          and os.path.basename(p).lower() != "supplementary_v3.zip"]
PILOT_ZIP = sorted(_cands, key=lambda p: ("10271" not in os.path.basename(p), p))[0] if _cands else None
if len(_cands) > 1:
    print("several candidate pilot archives; using the first:", *_cands, sep="\n  ")
# folder holding notebooks 00-08 (first one that contains them)
NOTEBOOK_DIR = next((d for d in (os.path.join(CFG.PROJECT, "notebooks"),
                                 os.path.join(CFG.DRIVE, "Colab Notebooks"))
                     if glob.glob(os.path.join(d, "0[0-8]_*.ipynb"))), None)
OUT_ZIP = os.path.join(CFG.PROJECT, "supplementary_v3.zip")
STAGE = "/content/v3_supplement_stage"
VERIFY_CLAIMS = True                                       # check every number quoted in the paper
# Identifying strings that must not appear anywhere in the archive (case-insensitive).
# They are read from a file OUTSIDE the archive, so this notebook itself never contains them.
# Create it once: one term per line (author names, institution, lab, grant numbers, e-mail).
TERMS_FILE = os.path.join(CFG.PROJECT, "anonymity_terms.txt")
if not os.path.exists(TERMS_FILE):
    raise SystemExit(f"Create {TERMS_FILE} with one identifying term per line, then re-run this cell.")
FORBIDDEN = [t.strip() for t in open(TERMS_FILE) if t.strip() and not t.startswith("#")]
assert len(FORBIDDEN) >= 3, "anonymity_terms.txt looks too short - list every author name and the institution"
print(f"anonymity terms: {len(FORBIDDEN)} loaded from {TERMS_FILE}")
print("pilot archive:", PILOT_ZIP or "NOT FOUND - upload the previous supplementary zip to MyDrive/TMLR_v3/ "
      "(any name containing 'Supplementary'); pilot/ will be omitted")
REQUIRE_PILOT = True    # the build stops if pilot/ cannot be included
if REQUIRE_PILOT and not PILOT_ZIP:
    raise SystemExit("previous supplementary archive not found - upload it to MyDrive/TMLR_v3/ and re-run")
print("notebooks   :", NOTEBOOK_DIR or "NOT FOUND - put notebooks 00-08 in TMLR_v3/notebooks/ (notebooks/ will be omitted)")

## Optional (recommended): Google Drive's own record of the registration file
Asks the Drive API for `prereg.json`'s **creation time, modification time and MD5 checksum**, and nothing else
(no owner or user names). If the checksum matches the file being shipped, Google's own record shows that this exact
content existed at that time, which is stronger evidence than a file-system timestamp. Colab will ask you to sign
in once. The result goes to `registration/drive_metadata.json` in the archive and is anonymity-scanned like
everything else. Skip this cell if you prefer; the archive then records the file-system time only.

In [ ]:
import hashlib, json
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
svc = build("drive", "v3")
res = svc.files().list(q="name = 'prereg.json' and trashed = false",
                       fields="files(id, name, createdTime, modifiedTime, md5Checksum, size)").execute()
local_md5 = hashlib.md5(open(CFG.PREREG_PATH, "rb").read()).hexdigest()
cands = [f for f in res.get("files", []) if f.get("md5Checksum") == local_md5]
if not cands:
    print("no Drive file with the same content as", CFG.PREREG_PATH, "- found:", res.get("files"))
else:
    f = min(cands, key=lambda f: f["createdTime"])          # the original, if copies exist
    meta = {"name": f["name"], "createdTime": f["createdTime"], "modifiedTime": f["modifiedTime"],
            "md5Checksum": f["md5Checksum"], "size": f.get("size"), "md5_matches_local": True,
            "source": "Google Drive API v3 files.list (fields: createdTime, modifiedTime, md5Checksum, size)"}
    with open(os.path.join(CFG.PROJECT, "registration_drive_metadata.json"), "w") as fh:
        json.dump(meta, fh, indent=1)
    print(json.dumps(meta, indent=1))

## Build, scan and verify

In [ ]:
import build_supplement as B, report as R
for m in (B, R):
    importlib.reload(m)
assert os.path.exists(os.path.join(CFG.RESULTS_DIR, "battery.json")), "run notebook 04 first"
if not os.path.exists(os.path.join(CFG.RESULTS_DIR, "paper_tables.json")):
    R.build()                                              # notebook 05's output
T = os.path.join(CFG.PROJECT, "supplement_templates")
log = B.stage(STAGE, notebook_dir=NOTEBOOK_DIR, pilot_zip=PILOT_ZIP,
              readme=open(os.path.join(T, "README.md")).read(),
              requirements=open(os.path.join(T, "requirements.txt")).read())
tl = B.timeline(STAGE)
print("registration:", tl["registration_utc"], "| feature files:", tl["feature_files"],
      "| first saved:", tl["first_feature_saved_utc"], "| last saved:", tl["last_feature_saved_utc"])
print("analysis battery generated:", tl["analysis_battery_generated_utc"])
if tl.get("registration_drive_created_utc"):
    print("Drive record of prereg.json: created", tl["registration_drive_created_utc"],
          "| checksum matches shipped file:", tl.get("registration_drive_md5_matches_shipped_file"))
if tl["extracted_before_registration"] or tl["extraction_runs_before_registration"]:
    print("*** WARNING: some features predate the registration - the paper must disclose this (see timeline.json) ***")
    print(tl["extracted_before_registration"][:10], tl["extraction_runs_before_registration"][:10])
else:
    print(f"timeline: every feature postdates the registration ({tl['feature_files_with_timestamp']} by their own "
          f"timestamp, {tl['feature_files_bounded_by_manifest_build_time']} by their manifest's build time)"
          if tl["all_features_after_registration"]
          else f"timeline: {tl['feature_files_without_any_time']} feature files have no timestamp at all - see timeline.json")
hits = B.anonymity_scan(STAGE, FORBIDDEN)
if hits:
    for h in hits[:30]:
        print("IDENTIFYING:", h)
    raise SystemExit(f"{len(hits)} identifying string(s) found - fix them before building")
print("anonymity scan: clean")
man = B.write_manifest(STAGE, log)
size = B.zip_dir(STAGE, OUT_ZIP)
print(f"archive: {OUT_ZIP}  ({size / 1e6:.1f} MB, {man['n_files']} files)")
for n in log["notes"]:
    print("note:", n)
if log.get("notebooks_missing"):
    raise SystemExit("notebooks " + ", ".join(log["notebooks_missing"]) + " not found: copy them into "
                     "MyDrive/TMLR_v3/notebooks/ and re-run (the paper cites notebooks 07 and 08)")
if size > 100e6:
    print("WARNING: over 100 MB - check the size limit on the submission form")
rc, out, err = B.verify_archive(OUT_ZIP, () if VERIFY_CLAIMS else ("--no-claims",))
print(out[-6000:])
if rc != 0:
    print(err[-3000:])
    raise SystemExit("VERIFICATION FAILED in the clean copy - do not upload")
print("\nREADY TO UPLOAD:", OUT_ZIP)